In [1]:
import yfinance as yf
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
# let's download 5 years of Apple stock data
df= yf.download("AAPL" , start ='2019-01-01', end ='2024-01-01')
print(df.shape)
print(df.head())

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
#drops any missing data
print(df.isnull().sum())
df.dropna(inplace=True)

Price   Ticker
Close   AAPL      0
High    AAPL      0
Low     AAPL      0
Open    AAPL      0
Volume  AAPL      0
dtype: int64


In [ ]:
df_2022 =df["2022-01-01":"20212-12-31"]
df_range =df["2021-06-01":"2021-12-31"]
print(df.index)
print(df.index.year)
print(df.index.dayofweek)

DatetimeIndex(['2019-01-02', '2019-01-03', '2019-01-04', '2019-01-07',
               '2019-01-08', '2019-01-09', '2019-01-10', '2019-01-11',
               '2019-01-14', '2019-01-15',
               ...
               '2023-12-15', '2023-12-18', '2023-12-19', '2023-12-20',
               '2023-12-21', '2023-12-22', '2023-12-26', '2023-12-27',
               '2023-12-28', '2023-12-29'],
              dtype='datetime64[ns]', name='Date', length=1258, freq=None)
Index([2019, 2019, 2019, 2019, 2019, 2019, 2019, 2019, 2019, 2019,
       ...
       2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023],
      dtype='int32', name='Date', length=1258)
Index([2, 3, 4, 0, 1, 2, 3, 4, 0, 1,
       ...
       4, 0, 1, 2, 3, 4, 1, 2, 3, 4],
      dtype='int32', name='Date', length=1258)


In [ ]:
# Daily return : how much did the price change from yesterday?
df["daily_return"] = df["Close"].pct_change()
print(df['daily_return'].head(10))

Date
2019-01-02         NaN
2019-01-03   -0.099607
2019-01-04    0.042689
2019-01-07   -0.002226
2019-01-08    0.019063
2019-01-09    0.016982
2019-01-10    0.003196
2019-01-11   -0.009818
2019-01-14   -0.015037
2019-01-15    0.020467
Name: daily_return, dtype: float64


In [ ]:
def compute_rsi(series, window =14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = delta.clip(upper=0)

    #average gain and losses over 14 days
    avg_gain = gain.rolling(window=window).mean()
    avg_loss = -loss.rolling(window=window).mean()

    #formula
    rs = avg_gain/avg_loss
    rsi = 100 -(100/(1 + rs))

    return(rsi)

In [ ]:
#rolling averages (7 -21 - 50 days) for closing
df['SMA_7'] = df["Close"].rolling(window =7).mean()
df['SMA_21'] =df["Close"].rolling(window=21).mean()
df['SMA_50'] = df["Close"].rolling(window=50).mean()

# n-day rolling standard deviation (volatility proxy)
df["volatility_7"] = df['Close'].rolling(window=7).std()
df['volatility_21'] = df['Close'].rolling(window= 21).std()
df['volatility_50'] = df['Close'].rolling(window=50).std()

#bollinger bands
df['BB_upper'] = df['SMA_21'] + 2 * df['volatility_21']
df['BB_lower'] = df['SMA_21'] - 2 * df['volatility_21']

#Relative Strength index flag
df['RSI_14'] = compute_rsi(df['Close'])

# lagged features
df['return_lag1'] = df['daily_return'].shift(1)
df['return_lag5'] = df['daily_return'].shift(5)
df['return_lag10'] = df['daily_return'].shift(10)

#clean up after any missing values that were just computed
df.dropna(inplace=True)
print(df[['volatility_7','volatility_21','volatility_50','RSI_14','BB_upper','BB_lower','return_lag1','return_lag5','return_lag10']].head())

Empty DataFrame
Columns: [(volatility_7, ), (volatility_21, ), (volatility_50, ), (RSI_14, ), (BB_upper, ), (BB_lower, ), (return_lag1, ), (return_lag5, ), (return_lag10, )]
Index: []
